# Authors' original code: Mixup for Node Classification on Pubmed
**COMP8240 project, Jaya Sharma (60852518)**. This is the run log of the authors' released script.

This notebook runs the **original, unmodified** `mixup.py` and `graph_conv.py` from the authors' repository
<https://github.com/vanoracai/MixupForGraph> (Wang et al., WWW '21; MIT licence).

**Environment:** Google Colab, **CPU** runtime, Python 3.12, **PyTorch 2.6.0**, PyTorch Geometric 2.8.0, torch-scatter 2.1.2, torch-sparse 0.6.18.

*Outputs below are from the original run. Dead-end install attempts have been removed.*

## 1. Get the authors' code
The repository contains two code files: `mixup.py` (the training script) and `graph_conv.py` (the `GraphConv` layer). The script reads its data from `../data/Pubmed`, so a `data` folder is created next to the repository.

In [ ]:
!git clone https://github.com/vanoracai/MixupForGraph.git


Cloning into 'MixupForGraph'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 0), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 666.83 KiB | 16.67 MiB/s, done.


In [ ]:
%cd MixupForGraph


/content/MixupForGraph


In [ ]:
!ls

Fig  graph_conv.py  LICENSE  mixup.py  README.md


In [ ]:
%cd /content
!mkdir -p data
%cd MixupForGraph

/content
/content/MixupForGraph


## 2. Install an older software setup

`mixup.py` imports `torch_sparse` (line 12), and Colab's default PyTorch (2.11) has no ready-made `torch-sparse` package. Installing it there means compiling from source, which took 30+ minutes. So an older setup was installed instead:

1. **PyTorch 2.6.0** (replacing 2.11).
2. **PyTorch Geometric**, reinstalled on top of it.
3. **torch-scatter / torch-sparse**: ready-made packages for PyTorch 2.6 (CPU). These install in seconds.

*Note:* plain `pip install torch==2.6.0` installs the GPU build (`2.6.0+cu124`). It also runs on a CPU runtime; it is just a larger download.

In [ ]:
!pip install -q torch==2.6.0 torchvision==0.21.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 848.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
!pip install -q torch-geometric

In [ ]:
!pip install torch-scatter torch-sparse \
-f https://data.pyg.org/whl/torch-2.6.0+cpu.html

Looking in links: https://data.pyg.org/whl/torch-2.6.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.1/546.1 kB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.5 MB/s eta 0:00:00


### 2.1 Check the setup
The graph extensions import correctly, so the original code can run without any edits.

In [ ]:
import torch
import torch_geometric
from torch_sparse import SparseTensor
from torch_scatter import scatter_add

print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("Graph extensions working")

PyTorch: 2.6.0+cu124
PyG: 2.8.0.post1
Graph extensions working


## 3. Run the original script (unmodified)

Settings are hard-coded in `mixup.py`:

| Setting | Value |
|---|---|
| Dataset | Pubmed (19,717 nodes, 3 classes) |
| Split | one random 60/20/20 split, seed 0 |
| Model | 3 × `GraphConv`, hidden 256, dropout 0.4 |
| Mixup | λ ~ Beta(4, 4) (the paper text says α = 1) |
| Training | Adam, lr 0.001, 299 epochs; prints train/test accuracy each epoch |

### 3.1 Baseline: `python mixup.py` (no Mixup)

In [ ]:
%cd /content/MixupForGraph
!python mixup.py

/content/MixupForGraph
Processing...
Done!
Epoch: 01, Loss: 1.0988, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 02, Loss: 1.0903, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 03, Loss: 1.0825, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 04, Loss: 1.0751, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 05, Loss: 1.0675, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 06, Loss: 1.0607, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 07, Loss: 1.0555, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 08, Loss: 1.0523, Train Acc: 0.4241, Test Acc: 0.4353
Epoch: 09, Loss: 1.0503, Train Acc: 0.6478, Test Acc: 0.6575
Epoch: 10, Loss: 1.0452, Train Acc: 0.6062, Test Acc: 0.6192
Epoch: 11, Loss: 1.0370, Train Acc: 0.6012, Test Acc: 0.6144
Epoch: 12, Loss: 1.0264, Train Acc: 0.6146, Test Acc: 0.6293
Epoch: 13, Loss: 1.0143, Train Acc: 0.6403, Test Acc: 0.6483
Epoch: 14, Loss: 1.0005, Train Acc: 0.6527, Test Acc: 0.6623
Epoch: 15, Loss: 0.9853, Train Acc: 0.6560, Test Acc: 0.6623
Epoch: 16, Loss: 0.9672, Train Acc: 0.6571

### 3.2 Mixup: `python mixup.py --mixup`

In [ ]:
!python mixup.py --mixup

Epoch: 01, Loss: 1.0987, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 02, Loss: 1.0905, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 03, Loss: 1.0827, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 04, Loss: 1.0756, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 05, Loss: 1.0686, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 06, Loss: 1.0629, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 07, Loss: 1.0607, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 08, Loss: 1.0623, Train Acc: 0.3972, Test Acc: 0.4057
Epoch: 09, Loss: 1.0611, Train Acc: 0.6447, Test Acc: 0.6542
Epoch: 10, Loss: 1.0595, Train Acc: 0.6039, Test Acc: 0.6159
Epoch: 11, Loss: 1.0565, Train Acc: 0.6107, Test Acc: 0.6240
Epoch: 12, Loss: 1.0510, Train Acc: 0.6427, Test Acc: 0.6539
Epoch: 13, Loss: 1.0493, Train Acc: 0.6546, Test Acc: 0.6625
Epoch: 14, Loss: 1.0444, Train Acc: 0.6518, Test Acc: 0.6572
Epoch: 15, Loss: 1.0425, Train Acc: 0.6521, Test Acc: 0.6569
Epoch: 16, Loss: 1.0382, Train Acc: 0.6539, Test Acc: 0.6633
Epoch: 17, Loss: 1.0300,

## 4. Results of this run

| | Final epoch (299) test acc | Mean test acc, last 10 epochs | Final train acc |
|---|---|---|---|
| Baseline (GraphConv) | 87.17 % | 87.12 % | 99.99 % |
| Mixup + GraphConv (α = 4) | **88.79 %** | **88.82 %** | 95.01 % |
| Difference | +1.62 | +1.70 | |

**Observations**
- The original code runs end to end. On this split, Mixup improves test accuracy by about 1.6 points.
- Mixup's train accuracy stays lower (95 % vs 99.99 %) while its test accuracy is higher. This is consistent with Mixup acting as a regulariser that reduces overfitting.
- **Caveat:** this is **one split and one seed**, and the script reports test accuracy per epoch with no validation-based model selection.
  - In my main reproduction, the same configuration over **10 splits × 3 seeds** with validation-based selection gives a smaller average Pubmed gain of **+0.52**.
  - With the paper's described setup (GCN, α = 1), the gain is about zero.